# LSTM versus original Transformer: frozen-model matches

This notebook runs two saved players against each other. It never trains, changes weights, or executes the training cells in your other notebooks. Use the BasicEnv kernel, which has PyTorch installed.

Each player uses the same original MCTS implementation, shared search budget, and shared deck. Seats alternate. Draws remain -1 in the saved decision labels, while the result summary reports draws separately. Win rate means wins divided by all completed games.

Both checkpoint paths are configured. The saved LSTM and Transformer architectures and their shared 60-card deck were verified. The loader rechecks deck composition on every run and stops if you select incompatible checkpoints.

A two-game run checks functionality; it does not establish which architecture is stronger. For a meaningful comparison, also document training effort and evaluate more games. Native simulator randomness is not guaranteed to be reproducible merely by seeding Python.


## 1. Select paths and match settings

Set `TRANSFORMER_CHECKPOINT` to a full saved baseline with architecture and deck metadata. Adjust `MATCH_GAMES` only after a small trial. Both models use `SEARCH_COUNT` during matches, even if they were saved with different search settings.


In [6]:
from pathlib import Path
from collections import Counter
import contextlib
import gzip
import hashlib
import json
import math
import os
import random
import re
import sys
import uuid
import torch

PROJECT_DIR = Path(r"C:\Users\scept\Documents\Classes\EEE 591\Project")
SIM_ROOT = PROJECT_DIR / "the-pokemon-company-ptcg-ai-battle-challenge-playground" / "sample_submission" / "sample_submission" / "sample_submission"
OUTPUT_ROOT = Path(r"C:\Users\scept\OneDrive\Documents\ChatGPT\ML Project\runs\matchups")

# Your saved local LSTM. Change either path to select another baseline.
LSTM_CHECKPOINT = PROJECT_DIR / "runs" / "run_20261120" / "lstm_trial_v1.pth"
TRANSFORMER_CHECKPOINT = Path(r"C:\Users\scept\OneDrive\Documents\ChatGPT\ML Project\runs\run_20261120_1654\trans_trial_v1.pth") 
# Replace with Path(r"C:\...\rl_trial_v2.pth")

MATCH_GAMES = 10  # Start with two; even counts balance player seats.
SEARCH_COUNT = 10  # Shared MCTS search budget for BOTH models.
SAVE_DECISIONS = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert (SIM_ROOT / "cg" / "cg.dll").is_file(), "Simulator cg.dll not found"
if "cg" in sys.modules and Path(sys.modules["cg"].__file__).resolve().parent != (SIM_ROOT / "cg").resolve():
    raise RuntimeError("A different cg package is loaded. Restart the kernel first.")
sys.dont_write_bytecode = True
if str(SIM_ROOT) not in sys.path:
    sys.path.insert(0, str(SIM_ROOT))

from cg.api import (
    AreaType, Card, Observation, OptionType, PlayerState, Pokemon,
    SearchState, SelectContext, all_attack, all_card_data,
    search_begin, search_end, search_step, to_observation_class,
)
from cg.game import battle_start, battle_finish, battle_select
print("Python:", sys.executable)
print("Device:", device)
print("Simulator:", SIM_ROOT / "cg")
print("Shared search budget:", SEARCH_COUNT)


Python: c:\Users\scept\miniconda3\envs\BasicEnv\python.exe
Device: cpu
Simulator: C:\Users\scept\Documents\Classes\EEE 591\Project\the-pokemon-company-ptcg-ai-battle-challenge-playground\sample_submission\sample_submission\sample_submission\cg
Shared search budget: 10


## 2. Define both models and the shared feature/search code

These definitions are copied from your current LSTM and original local Transformer notebooks. No model is randomly initialized here and no training loop runs. The `MyModel` class is the original Transformer. Although some helper type hints name `MyModel`, both models work because they accept the same six inputs and return the same `(value, action_scores)` outputs.


In [7]:
all_card = all_card_data()

card_table = {c.cardId:c for c in all_card}

card_count = max(all_card, key=lambda c: c.cardId).cardId + 1

attack_count = max(all_attack(), key=lambda a: a.attackId).attackId + 1

num_words_encoder = 24

encoder_size = max(22000, 17 * card_count + 43)

decoder_main_feature = 8

decoder_attack_offset = 14

decoder_card_offset = decoder_attack_offset + attack_count

decoder_size = decoder_card_offset + (1 + decoder_main_feature + SelectContext.RECOVER_SPECIAL_CONDITION) * card_count

class DecoderLayer(torch.nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_feedforward: int):
        super(DecoderLayer, self).__init__()

        self.attention = torch.nn.MultiheadAttention(d_model, num_heads)
        self.fc1 = torch.nn.Linear(d_model, d_feedforward)
        self.fc2 = torch.nn.Linear(d_feedforward, d_model)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor) -> torch.Tensor:
        y, _ = self.attention(x, encoder_out, encoder_out, need_weights=False)
        res = self.norm1(x + y)
        y = self.fc1(res)
        y = torch.nn.functional.relu(y)
        y = self.fc2(y)
        return self.norm2(res + y)

class MyModel(torch.nn.Module):
    def __init__(self,
                 d_model: int,
                 num_heads: int,
                 d_feedforward: int,
                 num_layers_encoder: int,
                 num_layers_decoder: int
    ):
        super(MyModel, self).__init__()

        self.d_model = d_model

        self.encoder_bag = torch.nn.EmbeddingBag(encoder_size, d_model, mode="sum")
        encoder_layer = torch.nn.TransformerEncoderLayer(d_model, num_heads, d_feedforward, 0)
        self.encoder = torch.nn.TransformerEncoder(encoder_layer, num_layers_encoder, enable_nested_tensor=False)
        self.encoder_fc = torch.nn.Linear(d_model, 1)
        self.decoder_bag = torch.nn.EmbeddingBag(decoder_size, d_model, mode="sum")
        self.decoder = torch.nn.ModuleList()
        for _ in range(num_layers_decoder):
            self.decoder.append(DecoderLayer(d_model, num_heads, d_feedforward))
        self.decoder_fc = torch.nn.Linear(d_model, 1)

    def forward(self,
                index_encoder: torch.Tensor,
                value_encoder: torch.Tensor,
                offset_encoder: torch.Tensor,
                index_decoder: torch.Tensor,
                value_decoder: torch.Tensor,
                offset_decoder: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        v = self.encoder_bag(index_encoder, offset_encoder, value_encoder)
        v = v.reshape(-1, num_words_encoder, self.d_model).transpose(0, 1)
        batch_size = v.size(1)
        encoder_out = self.encoder(v)
        v = self.encoder_fc(encoder_out)
        v = torch.tanh(v.mean(0))

        p = self.decoder_bag(index_decoder, offset_decoder, value_decoder)
        p = p.reshape(batch_size, -1, self.d_model).transpose(0, 1)
        for layer in self.decoder:
            p = layer(p, encoder_out)
        p = self.decoder_fc(p)
        p = p.transpose(0, 1).view(batch_size, -1)
        p = torch.tanh(p)
        return (v, p)

class SparseVector:
    index: list[int]
    value: list[float]
    offset: list[int]
    pos: int

    def __init__(self):
        self.index = []
        self.value = []
        self.offset = []
        self.pos = 0

    def add(self, index: int, value: float | int | bool):
        value = float(value)
        if value != 0.0:
            self.index.append(self.pos + index)
            self.value.append(value)

    def add_pos(self, pos: int):
        self.pos += pos

    def add_single(self, value: float | int | bool):
        value = float(value)
        if value != 0.0:
            self.index.append(self.pos)
            self.value.append(value)
        self.pos += 1

    def word_start(self):
        self.offset.append(len(self.index))

def add_card(sv: SparseVector, card: Card | Pokemon | None):
    if card != None:
        sv.add(card.id, 1)
    sv.add_pos(card_count)

def add_cards(sv: SparseVector, cards: list[Card] | None, value: float):
    if cards != None:
        for card in cards:
            sv.add(card.id, value)
    sv.add_pos(card_count)

def add_pokemon(sv: SparseVector, poke: Pokemon | None):
    if poke == None:
        sv.add_single(1)
        sv.add_pos(1 + 3 * card_count)
    else:
        sv.add_single(0)
        sv.add_single(poke.hp / 400)
        add_card(sv, poke)
        add_cards(sv, poke.tools, 1.0)
        add_cards(sv, poke.energyCards, 0.5)

def add_player(sv: SparseVector, ps: PlayerState):
    sv.add_single(ps.deckCount / 60)
    sv.add_single(len(ps.discard) / 60)
    sv.add_single(ps.handCount / 8)
    sv.add_single(len(ps.bench) / 5)
    sv.add(len(ps.prize), 1)
    sv.add_pos(7)

    sv.add_single(ps.poisoned)
    sv.add_single(ps.burned)
    sv.add_single(ps.asleep)
    sv.add_single(ps.paralyzed)
    sv.add_single(ps.confused)

    add_cards(sv, ps.discard, 0.25)

def get_encoder_input(obs: Observation, your_deck: list[int]) -> SparseVector:
    your_index = obs.current.yourIndex
    state = obs.current

    sv = SparseVector()
    for i in range(2):
        ps = state.players[i ^ your_index]
        for j in range(8): # For bench
            sv.word_start()
            pos = sv.pos
            if j < len(ps.bench):
                add_pokemon(sv, ps.bench[j])
            else:
                add_pokemon(sv, None)
            if j != 7:  # Not last
                sv.pos = pos  # Return to the previous position

    for i in range(2):
        ps = state.players[i ^ your_index]
        sv.word_start()
        if 0 < len(ps.active):
            add_pokemon(sv, ps.active[0])
        else:
            add_pokemon(sv, None)

    for i in range(2):
        ps = state.players[i ^ your_index]
        sv.word_start()
        add_player(sv, ps)

    sv.word_start()
    add_cards(sv, state.players[your_index].hand, 0.25)

    sv.word_start()
    for id in your_deck:
        sv.add(id, 0.25)
    sv.add_pos(card_count)

    sv.word_start()
    add_cards(sv, state.stadium, 1.0)

    sv.word_start()
    sv.add_single(1)
    sv.add_single(state.turn / 10)
    sv.add_single(state.firstPlayer == your_index)
    return sv

def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:
            return obs.select.deck[index]
        case AreaType.HAND:
            return ps.hand[index]
        case AreaType.DISCARD:
            return ps.discard[index]
        case AreaType.ACTIVE:
            return ps.active[index]
        case AreaType.BENCH:
            return ps.bench[index]
        case AreaType.PRIZE:
            return ps.prize[index]
        case AreaType.STADIUM:
            return obs.current.stadium[index]
        case AreaType.LOOKING:
            return obs.current.looking[index]
        case _:
            return None

def decoder_main(sv: SparseVector, feature_index: int, card: Card | Pokemon | None):
    if card != None:
        sv.add(decoder_card_offset + feature_index * card_count + card.id, 1)

def decoder_card_id(sv: SparseVector, context: SelectContext, card_id: int):
    sv.add(decoder_card_offset + (decoder_main_feature + context) * card_count + card_id, 1)

def decoder_card(sv: SparseVector, context: SelectContext, card: Card | Pokemon | None):
    if card != None:
        decoder_card_id(sv, context, card.id)

def get_decoder_input(obs: Observation, actions: list[list[int]]) -> SparseVector:
    sv = SparseVector()
    your_index = obs.current.yourIndex
    ps = obs.current.players[your_index]
    context = obs.select.context
    for action in actions:
        sv.word_start()

        if len(action) == 0:
            sv.add(0, 1)
            continue

        for i in action:
            o = obs.select.option[i]
            match o.type:
                case OptionType.END:
                    sv.add(1, 1)
                case OptionType.YES:
                    sv.add(2, 1)
                case OptionType.NO:
                    sv.add(3, 1)
                case OptionType.SPECIAL_CONDITION:
                    sv.add(4 + o.specialConditionType, 1)
                case OptionType.NUMBER:
                    sv.add(9 + min(o.number, 4), 1)
                case OptionType.ATTACK:
                    sv.add(decoder_attack_offset + o.attackId, 1)
                case OptionType.PLAY:
                    decoder_main(sv, 0, ps.hand[o.index])
                case OptionType.ATTACH:
                    decoder_main(sv, 1, get_card(obs, o.area, o.index, your_index))
                    decoder_main(sv, 2, get_card(obs, o.inPlayArea, o.inPlayIndex, your_index))
                case OptionType.EVOLVE:
                    decoder_main(sv, 3, get_card(obs, o.area, o.index, your_index))
                    decoder_main(sv, 4, get_card(obs, o.inPlayArea, o.inPlayIndex, your_index))
                case OptionType.ABILITY:
                    decoder_main(sv, 5, get_card(obs, o.area, o.index, your_index))
                case OptionType.DISCARD:
                    decoder_main(sv, 6, get_card(obs, o.area, o.index, your_index))
                case OptionType.RETREAT:
                    decoder_main(sv, 7, ps.active[0])
                case OptionType.CARD:
                    decoder_card(sv, context, get_card(obs, o.area, o.index, o.playerIndex))
                case OptionType.TOOL_CARD:
                    card = get_card(obs, o.area, o.index, o.playerIndex)
                    decoder_card(sv, context, card.tools[o.toolIndex])
                case OptionType.ENERGY_CARD | OptionType.ENERGY:
                    card = get_card(obs, o.area, o.index, o.playerIndex)
                    decoder_card(sv, context, card.energyCards[o.energyIndex])
                case OptionType.SKILL:
                    decoder_card_id(sv, context, o.cardId)

    return sv

def eval_nn(sv_enc: SparseVector, sv_dec:SparseVector, model: MyModel) -> tuple[float, list[float]]:
    device = next(model.parameters()).device
    value, policy = model(
        torch.tensor(sv_enc.index, dtype=torch.int32, device=device),
        torch.tensor(sv_enc.value, dtype=torch.float32, device=device),
        torch.tensor(sv_enc.offset, dtype=torch.int32, device=device),
        torch.tensor(sv_dec.index, dtype=torch.int32, device=device),
        torch.tensor(sv_dec.value, dtype=torch.float32, device=device),
        torch.tensor(sv_dec.offset, dtype=torch.int32, device=device))

    return (value.tolist()[0][0], policy.tolist()[0])

class LearnSample:
    def __init__(self, value: float, policy: list[float], sv_enc: SparseVector, sv_dec:SparseVector):
        self.value = value # Encoder output
        self.policy = policy # Decoder output
        self.sv_enc = sv_enc
        self.sv_dec = sv_dec

class Child:
    node: 'Node | None'
    select: list[int] # Selected option indices
    prob: float # Probability

    def __init__(self, select: list[int], prob: float):
        self.node = None
        self.select = select
        self.prob = prob

class Node:
    value: float # Self value
    total: float # Total value
    visit: int # Visit count
    parent: 'Node | None' # Parent node
    children: list[Child]
    state: SearchState # Search State of this node

    def __init__(self, parent: 'Node | None', state: SearchState):
        self.value = -2.0
        self.total = 0.0
        self.visit = 0
        self.parent = parent
        self.children = []
        self.state = state

    # Backpropagation value
    def backprop(self, value: float):
        self.total += value
        self.visit += 1
        if self.parent != None:
            self.parent.backprop(value)

def create_node(parent: Node | None,
                search_state: SearchState,
                your_index: int,
                your_deck: list[int],
                model: MyModel
    ) -> tuple[Node, LearnSample | None]:
    node = Node(parent, search_state)

    obs = search_state.observation
    state = obs.current
    if state.result >= 0:
        # Battle finished
        if state.result == 2:
            node.value = 0
        elif state.result == your_index:
            node.value = 1
        else:
            node.value = -1
        node.backprop(node.value)
        sample = None
    else:
        actions = []
        indices = list(range(obs.select.maxCount))
        for _ in range(64):
            actions.append(indices.copy())
            for i in range(len(indices)):
                index = len(indices) - i - 1
                if indices[index] < len(obs.select.option) - i - 1:
                    indices[index] += 1
                    for j in range(index+1, len(indices)):
                        indices[j] = indices[j - 1] + 1
                    break
            else:
                break

        sv_enc = get_encoder_input(obs, your_deck)
        sv_dec = get_decoder_input(obs, actions)
        value, policy = eval_nn(sv_enc, sv_dec, model)
        v = value
        if state.yourIndex != your_index:
            v = -v
        node.value = v
        node.backprop(v)

        sum = 0.0
        for i in range(len(policy)):
            p = math.exp(policy[i] * 10.0)
            node.children.append(Child(actions[i], p))
            sum += p
        for c in node.children:
            c.prob /= sum
        sample = LearnSample(value, policy, sv_enc, sv_dec)

    return (node, sample)

def mcts_agent(obs_dict: dict, your_deck: list[int], model: MyModel) -> tuple[list[int], LearnSample]:
    obs = to_observation_class(obs_dict)
    your_index = obs.current.yourIndex
    state = obs.current
    active = state.players[1 - your_index].active
    search_state = search_begin(
        obs,
        your_deck=random.sample(your_deck, state.players[your_index].deckCount), # Randomly select from deck.
        your_prize=random.sample(your_deck, len(state.players[your_index].prize)), # Randomly select from deck.
        opponent_deck=[1072] * state.players[1 - your_index].deckCount, # Fill with Snorlax (There is no deep meaning).
        opponent_prize=[1] * len(state.players[1 - your_index].prize), # Fill with Basic Energy (There is no deep meaning)
        opponent_hand=[1] * state.players[1 - your_index].handCount, # Fill with Basic Energy.
        opponent_active=[1072] if len(active) > 0 and active[0] == None else []) # Fill with Snorlax.
    root, sample = create_node(None, search_state, your_index, your_deck, model) # Create root node.

    # Search
    for _ in range(SEARCH_COUNT):
        current = root
        while True:
            value = -1e9
            c = 0.4 * math.sqrt(current.visit)
            for child in current.children:
                visit = 0
                if child.node == None:
                    v = current.total / current.visit
                else:
                    v = child.node.total / child.node.visit
                    visit = child.node.visit
                if current.state.observation.current.yourIndex != your_index:
                    v = -v
                v += c * child.prob / (1 + visit)
                if value < v:
                    value = v
                    next = child

            if next.node == None:
                search_state = search_step(current.state.searchId, next.select)
                next.node, _ = create_node(current, search_state, your_index, your_deck, model)
                break
            else:
                current = next.node
                if current.state.observation.current.result >= 0:
                    current.backprop(current.value)
                    break

    # Select the most visited node.
    max_child = None
    max_visit = -1
    min_value = 10
    for child in root.children:
        if child.node != None:
            if max_visit < child.node.visit:
                max_child = child
                max_visit = child.node.visit
            v = child.node.total / child.node.visit
            if min_value > v:
                min_value = v

    # Generate training data
    sample.value = root.total / root.visit
    for i in range(len(root.children)):
        child = root.children[i]
        v = sample.value
        if child.node == None:
            v = min_value - v - 0.03
        else:
            v = child.node.total / child.node.visit - v
        sample.policy[i] = max(-1.0, min(1.0, v))

    search_end()
    return (max_child.select, sample)

class LearnInput:
    index: list[int]
    value: list[float]
    offset: list[int]

    def __init__(self):
        self.index = []
        self.value = []
        self.offset = []

    def add(self, sv: SparseVector):
        count = len(self.index)
        self.index.extend(sv.index)
        self.value.extend(sv.value)
        for o in sv.offset:
            self.offset.append(o + count)

def random_agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    return random.sample(list(range(len(obs.select.option))), obs.select.maxCount)

def progress(count: int, text: str):
    current = 0
    while True:
        percent = 100 * current // count
        sys.stderr.write(f"\r{text} {percent}%   ")
        sys.stderr.flush()
        if(current >= count):
            sys.stderr.write("\n")
            sys.stderr.flush()
            break
        yield current
        current += 1

class LSTMModel(torch.nn.Module):
    def __init__(self,
                 embed_dim: int,
                 hidden_dim: int,
                 layer_dim: int,
                 output_dim: int,
    ):
        super(LSTMModel, self).__init__()

        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.embed_dim = embed_dim
        ## Replicating Architecture from Transformer

        ## Takes the feature inputs in their raw form (HP, energy,...) and changes
        ## them to embedding vectors according to the embed_dim set

        ## encoder_bag contains a trainable lookup table -- a table of weights, training updates them
        ## encoder_bag is for the current game state.
        ## This function is reflected in the name for the fully connect function
        ## for output v
        self.encoder_bag = torch.nn.EmbeddingBag(encoder_size, embed_dim, mode="sum")
        self.state_lstm = torch.nn.LSTM(input_size = embed_dim, hidden_size = hidden_dim, num_layers = layer_dim, batch_first = True)
        self.gamestate_fc = torch.nn.Linear(hidden_dim, out_features = output_dim)

        ## decoder_bag is for the candidate actions
        ## for output p
        self.decoder_bag = torch.nn.EmbeddingBag(decoder_size, embed_dim, mode = "sum")
        ## MLP NN for scoring the candidate actions
        self.candidates = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim + embed_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, output_dim),
            torch.nn.Tanh()
        )        


## * ======== Checkpoint
    def forward(self,
                index_encoder,
                value_encoder,
                offset_encoder,
                index_decoder,
                value_decoder,
                offset_decoder
    ):
 

## ? For now based on what found on basic LSTM online
## ? How to get multiple outputs?

        ## * v is the lstm output -- board state score
        ## Features of current board state getting embedded for weights
        v = self.encoder_bag(index_encoder, offset_encoder, value_encoder)

        ##  Embeddings to observations
        v = v.reshape(-1, num_words_encoder, self.embed_dim)

        lstm_output, (hn, cn) = self.state_lstm(v)

        ## ? Look into this
        board_summary = hn[-1]

        v = torch.tanh(self.gamestate_fc(board_summary))

## ***======
## ????????
        ## embedding candidate actions
        p = self.decoder_bag(index_decoder, offset_decoder, value_decoder)

        # 7. Organize candidates by observation.
        batch_size = board_summary.size(0)
        p = p.reshape(batch_size, -1, self.embed_dim )

        # 8. Provide the same board summary to every candidate.
        board_to_actions = board_summary.unsqueeze(1).expand(-1, p.size(1), -1)

        # 9. Combine board and action information.
        combined = torch.cat((board_to_actions, p), dim=-1)

        # 10. Predict one score per candidate.
        p = self.candidates(combined).squeeze(-1)

        return (v, p)


## 3. Define checkpoint loading and the match runner

The loader restores saved parameters, including older LSTM files whose architecture metadata was incorrect. Each model is frozen. The match runner selects the correct player on each decision, cleans up simulator resources, and saves progress after every completed game. Failed or interrupted games are not recorded as draws.


In [8]:
def load_match_player(path, expected_type):
    """Load a frozen baseline; never create a substitute with random weights."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if "model_state_dict" not in checkpoint or "sample_deck" not in checkpoint:
        raise ValueError("Select a full baseline checkpoint containing model_state_dict and sample_deck.")
    weights = checkpoint["model_state_dict"]
    actual_type = "LSTM" if "state_lstm.weight_hh_l0" in weights else "Transformer"
    if actual_type != expected_type:
        raise ValueError(f"Expected {expected_type}, but {path.name} contains {actual_type} weights.")
    if weights["encoder_bag.weight"].shape[0] != encoder_size or weights["decoder_bag.weight"].shape[0] != decoder_size:
        raise ValueError("Checkpoint feature vocabulary does not match this simulator/card data.")

    if actual_type == "LSTM":
        # Recover old LSTM checkpoints with erroneous Transformer metadata.
        config = {
            "embed_dim": weights["encoder_bag.weight"].shape[1],
            "hidden_dim": weights["state_lstm.weight_hh_l0"].shape[1],
            "layer_dim": sum(re.fullmatch(r"state_lstm\.weight_ih_l\d+", k) is not None for k in weights),
            "output_dim": weights["gamestate_fc.weight"].shape[0],
        }
        if config["output_dim"] != 1:
            raise ValueError("The LSTM must produce one value and one score per candidate.")
        player = LSTMModel(**config)
    else:
        # Attention head count is not recoverable from weight shapes alone.
        config = checkpoint.get("architecture")
        if not isinstance(config, (list, tuple)) or len(config) != 5:
            raise ValueError("Transformer checkpoint needs its five architecture settings.")
        player = MyModel(*config)

    player.load_state_dict(weights, strict=True)
    player.to(device)
    player.eval()
    player.requires_grad_(False)
    sha = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(block)
    info = {
        "model_type": actual_type,
        "checkpoint": str(path.resolve()),
        "checkpoint_sha256": sha.hexdigest(),
        "architecture": config,
        "training_deck": list(checkpoint["sample_deck"]),
        "saved_search_count": checkpoint.get("search_count"),
    }
    return player, info


def choose_shared_deck(lstm_info, transformer_info):
    """Require the same deck composition, independent of list ordering."""
    if Counter(lstm_info["training_deck"]) != Counter(transformer_info["training_deck"]):
        raise ValueError(
            "The checkpoints were saved with different decks. For this project's "
            "fixed-deck comparison, select or train a Transformer checkpoint "
            "using the same deck as the LSTM. Neither model was retrained."
        )
    deck = list(lstm_info["training_deck"])
    if len(deck) != 60:
        raise ValueError("The shared deck must contain 60 cards.")
    return deck


def play_matchup(lstm_player, transformer_player, lstm_info, transformer_info,
                 deck, num_games, output_root, max_decisions=5000, save_decisions=True):
    """Evaluate frozen networks with the same existing MCTS implementation."""
    if not isinstance(num_games, int) or num_games < 1:
        raise ValueError("num_games must be a positive integer; use an even number to balance seats.")
    if SEARCH_COUNT < 1:
        raise ValueError("SEARCH_COUNT must be positive.")
    if Counter(deck) != Counter(choose_shared_deck(lstm_info, transformer_info)):
        raise ValueError("Match deck differs from the shared training deck.")
    for player in (lstm_player, transformer_player):
        player.eval()
    match_dir = Path(output_root) / ("lstm_vs_transformer_" + uuid.uuid4().hex)
    match_dir.mkdir(parents=True, exist_ok=False)
    summary_path = match_dir / "summary.json"
    games = []
    counts = {"lstm_wins": 0, "transformer_wins": 0, "draws": 0}

    def persist(status, error=None):
        completed = len(games)
        summary = {
            "status": status, "requested_games": num_games,
            "completed_games": completed, **counts,
            "lstm_win_rate": counts["lstm_wins"] / completed if completed else None,
            "transformer_win_rate": counts["transformer_wins"] / completed if completed else None,
            "win_rate_definition": "model wins / all completed games; draws count as non-wins",
            "search_count_per_decision": SEARCH_COUNT,
            "deck": list(deck), "lstm": lstm_info, "transformer": transformer_info,
            "draw_outcome": -1, "games": games, "error": error,
        }
        summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        return summary

    persist("running")
    print("Match output:", match_dir)
    try:
        with torch.inference_mode():
            for game_number in range(num_games):
                lstm_seat = game_number % 2
                players = {lstm_seat: lstm_player, 1 - lstm_seat: transformer_player}
                names = {lstm_seat: "LSTM", 1 - lstm_seat: "Transformer"}
                game_id = uuid.uuid4().hex
                records = []
                decisions = 0
                observation, start_data = battle_start(deck, deck)
                if observation is None:
                    raise RuntimeError(f"Battle start failed: {start_data.errorType}")
                try:
                    while observation["current"]["result"] < 0:
                        if decisions >= max_decisions:
                            raise RuntimeError("Decision limit reached; this incomplete game is not counted as a draw.")
                        seat = observation["current"]["yourIndex"]
                        # Same algorithm and search budget; select the model by seat.
                        selected, _ = mcts_agent(observation, deck, players[seat])
                        if save_decisions:
                            records.append(json.dumps({
                                "game_id": game_id, "step": decisions, "player": seat,
                                "agent": names[seat], "observation": observation,
                                "selected_action": list(selected),
                            }))
                        observation = battle_select(selected)
                        decisions += 1
                    result = observation["current"]["result"]
                except BaseException:
                    # mcts_agent normally frees search state on success.
                    with contextlib.suppress(Exception):
                        search_end()
                    raise
                finally:
                    battle_finish()

                winner = "draw" if result == 2 else names[result]
                game_file = None
                if save_decisions:
                    game_file = game_id + ".jsonl.gz"
                    with gzip.open(match_dir / game_file, "wt", encoding="utf-8") as f:
                        for record in records:
                            row = json.loads(record)
                            row.update({"result": result, "winner": winner, "lstm_seat": lstm_seat,
                                        "outcome": 1 if result == row["player"] else -1})
                            f.write(json.dumps(row) + "\n")
                if winner == "draw":
                    counts["draws"] += 1
                elif winner == "LSTM":
                    counts["lstm_wins"] += 1
                else:
                    counts["transformer_wins"] += 1
                games.append({"game_id": game_id, "lstm_seat": lstm_seat,
                              "result": result, "winner": winner, "decisions": decisions, "file": game_file})
                summary = persist("running")
                print(f"Game {game_number + 1}/{num_games}: {winner}; "
                      f"LSTM {counts['lstm_wins']} wins, Transformer {counts['transformer_wins']} wins, "
                      f"{counts['draws']} draws | LSTM win rate: {100 * summary['lstm_win_rate']:.1f}%",
                      flush=True)
    except BaseException as exc:
        persist("interrupted", f"{type(exc).__name__}: {exc}")
        raise
    summary = persist("complete")
    print("Saved summary:", summary_path)
    return summary, match_dir


## 4. Load the players and confirm the shared deck

This stops with a clear message if a checkpoint is missing or incompatible. It checks deck composition rather than the order of cards in the stored list.


In [9]:
if TRANSFORMER_CHECKPOINT is None:
    raise ValueError(
        "Set TRANSFORMER_CHECKPOINT in section 1 to your trained Transformer .pth baseline. "
        "Choose the full saved Transformer baseline checkpoint."
    )

lstm_player, lstm_info = load_match_player(LSTM_CHECKPOINT, "LSTM")
transformer_player, transformer_info = load_match_player(TRANSFORMER_CHECKPOINT, "Transformer")
match_deck = choose_shared_deck(lstm_info, transformer_info)
print("LSTM:", lstm_info["checkpoint"])
print("Transformer:", transformer_info["checkpoint"])
print("Shared deck confirmed:", len(match_deck), "cards")
print("Both models are in evaluation mode; no optimizer is created.")


LSTM: C:\Users\scept\Documents\Classes\EEE 591\Project\runs\run_20261120\lstm_trial_v1.pth
Transformer: C:\Users\scept\OneDrive\Documents\ChatGPT\ML Project\runs\run_20261120_1654\trans_trial_v1.pth
Shared deck confirmed: 60 cards
Both models are in evaluation mode; no optimizer is created.


## 5. Play the matches

Both seats run MCTS: there is no random opponent. Even-numbered games place the LSTM in player 0; odd-numbered games place it in player 1 (using zero-based game numbering). This cell creates a separate results folder on every run.


In [11]:
MATCH_GAMES = 20
match_summary, match_directory = play_matchup(
    lstm_player, transformer_player,
    lstm_info, transformer_info,
    deck=match_deck,
    num_games=MATCH_GAMES,
    output_root=OUTPUT_ROOT,
    save_decisions=SAVE_DECISIONS,
)


Match output: C:\Users\scept\OneDrive\Documents\ChatGPT\ML Project\runs\matchups\lstm_vs_transformer_f8ad31616b304ed1a439b4c2a6679134
Game 1/20: Transformer; LSTM 0 wins, Transformer 1 wins, 0 draws | LSTM win rate: 0.0%
Game 2/20: Transformer; LSTM 0 wins, Transformer 2 wins, 0 draws | LSTM win rate: 0.0%
Game 3/20: Transformer; LSTM 0 wins, Transformer 3 wins, 0 draws | LSTM win rate: 0.0%
Game 4/20: Transformer; LSTM 0 wins, Transformer 4 wins, 0 draws | LSTM win rate: 0.0%
Game 5/20: Transformer; LSTM 0 wins, Transformer 5 wins, 0 draws | LSTM win rate: 0.0%
Game 6/20: Transformer; LSTM 0 wins, Transformer 6 wins, 0 draws | LSTM win rate: 0.0%
Game 7/20: Transformer; LSTM 0 wins, Transformer 7 wins, 0 draws | LSTM win rate: 0.0%
Game 8/20: Transformer; LSTM 0 wins, Transformer 8 wins, 0 draws | LSTM win rate: 0.0%
Game 9/20: LSTM; LSTM 1 wins, Transformer 8 wins, 0 draws | LSTM win rate: 11.1%
Game 10/20: Transformer; LSTM 1 wins, Transformer 9 wins, 0 draws | LSTM win rate: 10.0%


## 6. Inspect results

`summary.json` contains both checkpoint paths and hashes, deck, search count, game results, and win rates. If enabled, compressed JSONL files contain each player's observations and choices with agent labels and final outcomes. Keep reported evaluation games separate from future model-training datasets.


In [12]:
print("LSTM wins:", match_summary["lstm_wins"])
print("Transformer wins:", match_summary["transformer_wins"])
print("Draws:", match_summary["draws"])
print(f"LSTM win rate: {100 * match_summary['lstm_win_rate']:.1f}%")
print(f"Transformer win rate: {100 * match_summary['transformer_win_rate']:.1f}%")
print("Results folder:", match_directory)


LSTM wins: 5
Transformer wins: 15
Draws: 0
LSTM win rate: 25.0%
Transformer win rate: 75.0%
Results folder: C:\Users\scept\OneDrive\Documents\ChatGPT\ML Project\runs\matchups\lstm_vs_transformer_f8ad31616b304ed1a439b4c2a6679134
